# अनुच्छेद ०५ - एजेन्टिक RAG


## सेटअप

यस नोटबुकले Microsoft Agent Framework प्रयोग गरेर Agentic RAG (Retrieval-Augmented Generation) नमूनालाई देखाउँछ।

**पूर्वआवश्यकताहरू:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — तपाईंको Azure AI खोज सेवा इन्डप्वाइन्ट
- `AZURE_SEARCH_API_KEY` — तपाईंको Azure AI खोज API कुञ्जी
- वातावरण चरहरू मार्फत कन्फिगर गरिएको Azure OpenAI डिप्लोइमेन्ट
- Azure CLI प्रमाणित गरिएको (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Agentic RAG के हो?

परम्परागत RAG एक निश्चित पाइपलाइन अनुसरण गर्दछ: कागजातहरू पुनः प्राप्त गर्नुहोस्, त्यसपछि उत्तर सिर्जना गर्नुहोस्। **Agentic RAG** ले अगाडी बढेर एजेन्टलाई स्वायत्तता दिन्छ कि **कहिले** र **कसरी** जानकारी पुनः प्राप्त गर्ने निर्णय गर्न।

Agentic RAG सँग, एजेन्टले गर्न सक्छ:
- प्रश्नको उत्तर दिनुअघि पुनः प्राप्ति आवश्यक छ कि छैन भनेर **निर्णय गर्नु**
- कुन डेटा स्रोत वा उपकरण सोध्न **छान्नु**
- पुनः प्राप्त परिणामहरूको **मूल्याङ्कन गर्नु** र यदि पहिलो प्रयास अपर्याप्त छ भने फलो-अप पुनः प्राप्ति गर्नु
- बहु पुनः प्राप्ति चरणहरूबाट जानकारी **समायोजन गर्नु** र एक सुसंगत उत्तर तयार गर्नु

यसले एजेन्टलाई स्थिर retrieve-then-generate पाइपलाइनको तुलनामा बढी लचिलो र सटीक बनाउँछ।


## खोज उपकरण सिर्जना गर्दै

Agentic RAG मा, बाह्य डाटा स्रोतहरूलाई **उपकरण** को रूपमा ल्याप गर्नु हुन्छ जसलाई एजेण्ट आवश्यक पर्दा कल गर्न सक्छ। यसले एजेण्टलाई पुनःप्राप्तिलाई केवल अर्को क्रिया जस्तै लिन अनुमति दिन्छ, अनिवार्य चरण होइन।

तल हामीले एउटा यात्रा ज्ञान आधार परिभाषित गर्दैछौं र यसलाई एउटा उपकरणको रूपमा देखाउँदैछौं जुन एजेण्टले गन्तव्य जानकारी हेर्न कल गर्न सक्छ।


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG एजेन्ट निर्माण

अब हामी एउटा एजेन्ट सिर्जना गर्छौं जसलाई **जवाफ दिनुअघि सधैं जानकारी प्राप्त गर्न** निर्देशन दिइन्छ। एजेन्टले `search_travel_knowledge` उपकरण प्रयोग गर्दछ जसले यसको प्रतिक्रियालाई आफ्नै प्रशिक्षण डाटामा निर्भर नगरी ज्ञान आधारमा आधारित बनाउँछ।


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## आवर्ती पुन:प्राप्ति — मेकर-चेकर नमूना

एजेन्टिक RAG को एउटा प्रमुख फाइदा **आवर्ती पुन:प्राप्ति** हो। एजेन्टले आफ्नो प्रारम्भिक पत्ता लगाउने कुराहरूलाई पुष्टि, सुधार वा विस्तार गर्न धेरै पटक खोजी गर्न सक्छ — "मेकर-चेकर" कार्यप्रवाह जस्तै:

1. **मेकर चरण**: एजेन्टले प्रारम्भिक जानकारी पुन:प्राप्ति गर्छ र प्रतिक्रिया तयार गर्छ।
2. **चेकर चरण**: एजेन्टले विवरणहरू पुष्टि गर्न वा खाली ठाउँहरू भर्ने थप पुन:प्राप्ति गर्छ।

तल, एजेन्टलाई यस्तो प्रश्न सोधिन्छ जसले धेरै गन्तव्यहरूको तुलना गर्न आवश्यक हुन्छ, जसले यसलाई धेरै पटक खोजी गर्न उत्प्रेरित गर्छ।


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## सारांश

यस पाठमा तपाईंले Microsoft Agent Framework प्रयोग गरेर कसरी **Agentic RAG** प्रणाली निर्माण गर्ने सिक्नुभयो:

- **Agentic RAG** ले एजेन्टहरूलाई स्वायत्त रूपमा जानकारी ल्याउने निर्णय लिन दिन्छ, जसले ल्याउने प्रक्रिया स्थिर नभई गतिशील बनाउँछ।
- **डेटा स्रोतको रूपमा उपकरणहरू**: बाह्य ज्ञान आधारहरू (जस्तै Azure AI Search) उपकरणको रूपमा प्याकेज गरिन्छ जसलाई एजेन्टले प्रयोग गर्न सक्छ।
- **पुनरावृत्ति ल्याउने प्रक्रिया**: मेकर-चेकर ढाँचाले एजेन्टलाई बहु चरणको जानकारी ल्याउने प्रक्रिया गर्न अनुमति दिन्छ — खोज्ने, जाँच गर्ने, र सुधार गर्ने — अन्तिम उत्तर तयार गर्नु अघि।

उत्पादनमा, तपाईंले इन-मेमोरी `TRAVEL_KNOWLEDGE_BASE` लाई ठूला स्तरका यात्रा कागजात ल्याउनको लागि वास्तविक Azure AI Search सूचकांकले बदल्नुहुनेछ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज [Co-op Translator](https://github.com/Azure/co-op-translator) नामक एआई अनुवाद सेवा प्रयोग गरेर अनुवाद गरिएको हो। हामी शुद्धताका लागि प्रयास गर्दछौं तापनि, कृपया ध्यान दिनुहोस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धिहरू हुन सक्छन्। मूल भाषामा रहेको दस्तावेजलाई आधिकारिक स्रोत मान्नुपर्दछ। महत्त्वपूर्ण जानकारीका लागि पेशेवर मानव अनुवाद सिफारिश गरिएको छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलतफहमी वा गलत व्याख्याका लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
